# Patch desiname for older specprods

Need to patch `desiname` for `guadalupe`, `iron`, `loa` in light of [desiutil #211](https://github.com/desihub/desiutil/issues/211).

In [ ]:
import os
import sys
import numpy as np
from specprodDB import load as db
from sqlalchemy import or_
from desiutil.log import get_logger
from desiutil.names import radec_to_desiname

In [ ]:
specprod = 'guadalupe'
specprod = 'iron'
specprod = 'loa'

In [ ]:
db.log = get_logger()
postgresql = db.setup_db(schema=specprod, hostname='specprod-db.desi.lbl.gov', username='desi')

In [ ]:
schema = getattr(db, specprod)
Photometry = getattr(schema, 'Photometry')
Ztile = getattr(schema, 'Ztile')
Zpix = getattr(schema, 'Zpix')
for table in ('ztile', 'zpix'):
    Z = Ztile if table == 'ztile' else Zpix
    if table == 'ztile':
        q = db.dbSession.query(Ztile.id, Ztile.targetid, Ztile.tileid, Ztile.lastnight, Ztile.desiname,
                               Photometry.ra, Photometry.dec).filter(Photometry.targetid == Ztile.targetid).filter(or_(Photometry.ra < 0.1, Photometry.dec.between(-0.1, 0.1))).all()
    else:
        q = db.dbSession.query(Zpix.id, Zpix.targetid, Zpix.survey, Zpix.program, Zpix.desiname,
                               Photometry.ra, Photometry.dec).filter(Photometry.targetid == Zpix.targetid).filter(or_(Photometry.ra < 0.1, Photometry.dec.between(-0.1, 0.1))).all()
    ra = np.array([row[5] for row in q])
    dec = np.array([row[6] for row in q])
    new_desiname = radec_to_desiname(ra, dec)
    lines = list()
    for i, row in enumerate(q):
        lines.append(f"UPDATE {specprod}.{table} SET desiname = '{new_desiname[i]}' WHERE id = {row[0]};")
    with open(os.path.join(os.environ['SCRATCH'], f"patch_desiname_{specprod}.{table}.sql"), 'w') as SQL:
        SQL.write('\n'.join(lines) + '\n')